In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "python")))
from ribbon_graph_generator import generate_ribbon_graphs, get_disc_boundary
import ell_to_tau as et
import partition_function as pf
import importlib
importlib.reload(pf)
importlib.reload(et)

<module 'ell_to_tau' from '/Users/sam/Documents/research/string_notes/strebel_differential/python_code/covariant formalism/python/ell_to_tau.py'>

In [ ]:
import ell_to_tau as et
rgs = generate_ribbon_graphs(n_faces=1, n_edges=9)
data = get_disc_boundary(rgs[0])  # returns dict with edge_sequence, vertex_sequence, sewing

V=6, E=9, F=1, g=2
Using 2 base cubic graph(s)
Total valid rotation systems: 34992
  Base graph (6V, 9E): 72 automorphisms, 17496 rotation systems
  Base graph (6V, 9E): 12 automorphisms, 17496 rotation systems
Non-isomorphic ribbon graphs: 4


In [8]:
rgs[0]

([(1, 4), (1, 5), (1, 6), (2, 4), (2, 5), (2, 6), (3, 4), (3, 5), (3, 6)],
 [1, 2, 3, 4, 5, 6],
 {1: [0, 1, 2],
  2: [3, 4, 5],
  3: [6, 7, 8],
  4: [0, 3, 6],
  5: [1, 4, 7],
  6: [2, 8, 5]})

In [3]:
# Genus 1, F=1: theta graph (V=2, E=3)
rgs_g1 = generate_ribbon_graphs(n_faces=1, n_edges=3)
from ribbon_graph_generator import print_disc
for i, rg in enumerate(rgs_g1):
    print_disc(rg, i + 1)

V=2, E=3, F=1, g=1
Using 1 base cubic graph(s)
Total valid rotation systems: 18
  Base graph (2V, 3E): 2 automorphisms, 18 rotation systems
Non-isomorphic ribbon graphs: 1

Graph 1: V=2, E=3, g=1, boundary=6

Counterclockwise boundary:
 Seg  Vertex  Edge   Sewed to
--------------------------------
   1       1     1          4
   2       2     2          5
   3       1     3          6
   4       2     1          1
   5       1     2          2
   6       2     3          3

Edge sequence:   [1, 2, 3, 1, 2, 3]
Vertex sequence: [1, 2, 1, 2, 1, 2]

Sewing pairs:
  Edge 1: segment 1 <-> segment 4
  Edge 2: segment 2 <-> segment 5
  Edge 3: segment 3 <-> segment 6


In [25]:
# Genus-1 sanity check for the raw general-genus constructor.
# Note: periods_improved only works with make_cyl_eqn_improved, because it expects
# f(z) to return (singular_factor, regular_polynomial).  The raw constructors instead
# return a bare polynomial, so for those we compare using periods_given_f.
import numpy as np

rg1 = rgs_g1[0]
L_g1 = 26*81
l1_g1, l2_g1 = 3*81, 1*81
ell_g1 = [l1_g1, l2_g1, L_g1 // 2 - l1_g1 - l2_g1]

f_ref = et.make_cyl_eqn(L_g1, l1_g1, l2_g1)
P_ref = et.periods_given_f(L_g1, l1_g1, l2_g1, f_ref)
tau_ref = P_ref[1] / P_ref[0]

P_imp = et.periods_improved(L_g1, l1_g1, l2_g1, et.make_cyl_eqn_improved(L_g1, l1_g1, l2_g1))
tau_imp = P_imp[1] / P_imp[0]

forms_g1 = et.make_cyl_eqn_general_genus(rg1, ell_g1)

print('ell_g1 =', ell_g1)
print('reference periods from make_cyl_eqn =', P_ref)
print('reference tau from make_cyl_eqn =', tau_ref)
print('reference tau from make_cyl_eqn_improved / periods_improved =', tau_imp)
print('number of candidate forms from make_cyl_eqn_general_genus =', len(forms_g1))

c_ref = np.asarray(f_ref.coeffs, dtype=np.complex128)
c_ref /= np.linalg.norm(c_ref)

for i, f in enumerate(forms_g1):
    P = et.periods_given_f(L_g1, l1_g1, l2_g1, f)
    tau = P[1] / P[0]
    c = np.asarray(f.coeffs, dtype=np.complex128)
    c /= np.linalg.norm(c)
    phase = np.vdot(c_ref, c)
    c_aligned = c if abs(phase) == 0 else c * phase / abs(phase)
    coeff_err = np.linalg.norm(c_ref - c_aligned)
    print(f'candidate {i}: periods = {P}')
    print(f'candidate {i}: tau = {tau}, |tau - tau_ref| = {abs(tau - tau_ref):.6g}, coeff_err = {coeff_err:.6g}')


Smallest singular values: [3.56443892e+01 2.52775485e+01 2.52775485e+01 6.34012258e-12
 5.34044395e-14]
ell_g1 = [243, 81, 729]
reference periods from make_cyl_eqn = (np.complex128(-2.2170384863546633-0.6787125246758418j), np.complex128(-1.4400922708856312-2.129011499106462j), np.complex128(0.7769049327839774-1.4508598350784152j))
reference tau from make_cyl_eqn = (0.8626867755951558+0.6961968360665154j)
reference tau from make_cyl_eqn_improved / periods_improved = (0.8627673642999393+0.6963712264258495j)
number of candidate forms from make_cyl_eqn_general_genus = 2
candidate 0: periods = (np.complex128(-0.16101881217524303-0.10705965156762969j), np.complex128(-0.10491072908786914-0.1349867093878086j), np.complex128(0.04625746103930821-0.22212451509390885j))
candidate 0: tau = (0.8383314931505048+0.2809313472249042j), |tau - tau_ref| = 0.415979, coeff_err = 1.38894
candidate 1: periods = (np.complex128(1.0201148901341963+0.6642874427174026j), np.complex128(0.6587600041309364+0.86380739

In [17]:
# Genus-1 reduction test for the Hermitian block solver inside
# make_cyl_eqn_improved_higher_genus.
rg1 = rgs_g1[0]
L_g1 = 6*81
l1_g1, l2_g1 = 1 * 81, 1 * 81
ell_g1 = [l1_g1, l2_g1, L_g1 // 2 - l1_g1 - l2_g1]

forms_hi = et.make_cyl_eqn_improved_higher_genus(rg1, ell_g1)
f_hi = forms_hi[0]
f_ref = et.make_cyl_eqn_improved(L_g1, l1_g1, l2_g1)

P_hi = et.periods_improved(L_g1, l1_g1, l2_g1, f_hi)
P_ref = et.periods_improved(L_g1, l1_g1, l2_g1, f_ref)
tau_hi = P_hi[1] / P_hi[0]
tau_ref = P_ref[1] / P_ref[0]

print('ell_g1 =', ell_g1)
print('number of forms from make_cyl_eqn_improved_higher_genus =', len(forms_hi))
print('higher-genus solver =', f_hi.solver)
print('higher-genus pivot indices =', f_hi.pivot_indices)
print('higher-genus improved singular values tail =', f_hi.singular_values[-5:])
print('periods from make_cyl_eqn_improved_higher_genus =', P_hi)
print('periods from make_cyl_eqn_improved =', P_ref)
print('tau from make_cyl_eqn_improved_higher_genus =', tau_hi)
print('tau from make_cyl_eqn_improved =', tau_ref)
print('|tau_hi - tau_ref| =', abs(tau_hi - tau_ref))


ell_g1 = [81, 81, 81]
number of forms from make_cyl_eqn_improved_higher_genus = 1
higher-genus solver = hermitian_block
higher-genus pivot indices = (241,)
higher-genus improved singular values tail = [1.75017471e+01 1.75016771e+01 1.74985691e+01 1.39026795e+01
 4.60659289e-13]
periods from make_cyl_eqn_improved_higher_genus = (np.complex128(8.534284390293578e-13+8.668368397418055e-15j), np.complex128(6.405154184818684e-13+5.7520461636044956e-15j), np.complex128(2.129130205474894e-13+2.916322233813559e-15j))
periods from make_cyl_eqn_improved = (np.complex128(1.6693690117834517+0.9638106483299319j), np.complex128(-1.532107773982716e-14+1.9276212966599102j), np.complex128(1.6693690117834672-0.9638106483299784j))
tau from make_cyl_eqn_improved_higher_genus = (0.7505113892973211-0.0008831004578894707j)
tau from make_cyl_eqn_improved = (0.4999999999999558+0.866025403784435j)
|tau_hi - tau_ref| = 0.9023781418537133


In [ ]:
f_basis = et.make_cyl_eqn_general_genus(rgs[0],[11,11,11,11,11,11,11,11,11])

Smallest singular values: [7.87796787e+00 7.69076153e+00 4.24025336e-01 4.67695784e-02
 8.83723929e-15]


[<function ell_to_tau.make_cyl_eqn_general_genus.<locals>._make_f.<locals>.f(z)>]

In [19]:
# Genus-2 test: print the two candidate holomorphic one-forms from
# make_cyl_eqn_improved_higher_genus.
forms_g2_hi = et.make_cyl_eqn_improved_higher_genus(rgs[0], [11]*9)
print('number of forms =', len(forms_g2_hi))
for i, f in enumerate(forms_g2_hi):
    print(f'form {i}: solver = {f.solver}, pivots = {f.pivot_indices}')
    #print(f'form {i}: coeffs = {f.coeffs}')


number of forms = 2
form 0: solver = hermitian_block, pivots = (1, 98)
form 1: solver = hermitian_block, pivots = (1, 98)


In [22]:
# Disc-frame homology basis from edge chords for genus 1 and 2.
def print_edge_homology_basis(label, rg):
    data = et.find_edge_homology_basis(rg)
    J = data['intersection_matrix']
    basis_pairs = data['basis_pairs']

    print(label)
    print('genus =', data['genus'])
    print('boundary word =', data['boundary_word'])
    print('edge positions =', data['edge_positions'])
    print('basis pairs:')
    for i, pair in enumerate(basis_pairs, start=1):
        print(f"  alpha_{i} = {pair['alpha']}, beta_{i} = {pair['beta']}")

    alpha_edges = [pair['alpha'][0][0] for pair in basis_pairs]
    beta_edges = [pair['beta'][0][0] for pair in basis_pairs]
    beta_signs = np.diag([pair['beta'][0][1] for pair in basis_pairs])

    print('symplectic alpha-beta block =')
    print(J[np.ix_(alpha_edges, beta_edges)] @ beta_signs)
    print('full submatrix on (alpha_1,...,alpha_g,beta_1,...,beta_g) =')
    print(J[np.ix_(alpha_edges + beta_edges, alpha_edges + beta_edges)])
    print()

print_edge_homology_basis('Genus 1 theta graph', rgs_g1[0])
print_edge_homology_basis('Genus 2 first one-face graph', rgs[0])


Genus 1 theta graph
genus = 1
boundary word = (0, 1, 2, 0, 1, 2)
edge positions = {0: (0, 3), 1: (1, 4), 2: (2, 5)}
basis pairs:
  alpha_1 = [(0, 1)], beta_1 = [(1, 1)]
symplectic alpha-beta block =
[[1]]
full submatrix on (alpha_1,...,alpha_g,beta_1,...,beta_g) =
[[ 0  1]
 [-1  0]]

Genus 2 first one-face graph
genus = 2
boundary word = (0, 3, 4, 7, 8, 5, 3, 6, 7, 1, 2, 8, 6, 0, 1, 4, 5, 2)
edge positions = {0: (0, 13), 1: (9, 14), 2: (10, 17), 3: (1, 6), 4: (2, 15), 5: (5, 16), 6: (7, 12), 7: (3, 8), 8: (4, 11)}
basis pairs:
  alpha_1 = [(0, 1)], beta_1 = [(1, 1)]
  alpha_2 = [(3, 1)], beta_2 = [(7, 1)]
symplectic alpha-beta block =
[[1 0]
 [0 1]]
full submatrix on (alpha_1,...,alpha_g,beta_1,...,beta_g) =
[[ 0  0  1  0]
 [ 0  0  0  1]
 [-1  0  0  0]
 [ 0 -1  0  0]]



In [26]:
# Genus-1 regression test for period_matrix.
g1_period_cases = [
    ('generic', 26 * 81, 3 * 81, 1 * 81, 5e-5),
    ('equal lengths', 6 * 81, 1 * 81, 1 * 81, 1e-10),
]

rg1 = rgs_g1[0]
for label, L_g1, l1_g1, l2_g1, tol in g1_period_cases:
    ell_g1 = [l1_g1, l2_g1, L_g1 // 2 - l1_g1 - l2_g1]
    f_g1 = et.make_cyl_eqn_improved(L_g1, l1_g1, l2_g1)

    tau_pm = et.period_matrix(L=L_g1, l1=l1_g1, l2=l2_g1)
    tau_pm_forms = et.period_matrix(forms=[f_g1], ribbon_graph=rg1, ell_list=ell_g1)

    P_ref = et.periods_improved(L_g1, l1_g1, l2_g1, f_g1)
    tau_ref = P_ref[1] / P_ref[0]

    err_pm = abs(tau_pm - tau_ref)
    err_pm_forms = abs(tau_pm_forms - tau_ref)
    err_between_entries = abs(tau_pm - tau_pm_forms)

    print(label)
    print('ell_g1 =', ell_g1)
    print('tau from period_matrix(L, l1, l2) =', tau_pm)
    print('tau from period_matrix(forms=...) =', tau_pm_forms)
    print('tau from periods_improved =', tau_ref)
    print('|tau_pm - tau_ref| =', err_pm)
    print('|tau_pm_forms - tau_ref| =', err_pm_forms)
    print('|tau_pm - tau_pm_forms| =', err_between_entries)
    print()

    assert err_pm < tol, f'{label}: period_matrix(L,l1,l2) mismatch {err_pm} >= {tol}'
    assert err_pm_forms < tol, f'{label}: period_matrix(forms=...) mismatch {err_pm_forms} >= {tol}'
    assert err_between_entries < 1e-12, (
        f'{label}: two period_matrix entry points disagree by {err_between_entries}'
    )

print('Genus-1 period_matrix tests passed.')


generic
ell_g1 = [243, 81, 729]
tau from period_matrix(L, l1, l2) = (0.862746536687166+0.6963780146315686j)
tau from period_matrix(forms=...) = (0.862746536687166+0.6963780146315686j)
tau from periods_improved = (0.8627673642999393+0.6963712264258495j)
|tau_pm - tau_ref| = 2.1905916796973968e-05
|tau_pm_forms - tau_ref| = 2.1905916796973968e-05
|tau_pm - tau_pm_forms| = 0.0

equal lengths
ell_g1 = [81, 81, 81]
tau from period_matrix(L, l1, l2) = (0.4999999999999998+0.8660254037844385j)
tau from period_matrix(forms=...) = (0.4999999999999998+0.8660254037844385j)
tau from periods_improved = (0.4999999999999558+0.866025403784435j)
|tau_pm - tau_ref| = 4.409933868605904e-14
|tau_pm_forms - tau_ref| = 4.409933868605904e-14
|tau_pm - tau_pm_forms| = 0.0

Genus-1 period_matrix tests passed.


In [ ]:
# Genus-2 candidate separating degeneration test with a custom homology basis.
# Reload ell_to_tau so this cell sees the latest custom-cycle validation code.
import importlib
et = importlib.reload(et)

# For graph rgs[1], edges 0,1,2 form the left torus block, edges 3,4,5 form
# the right torus block, and edges 6,7,8 are the connector/neck edges.
# To see the two-torus limit, scale all three connector lengths together and
# compute periods on cycles supported on the left/right torus blocks.
# The naive mirrored choice (0+2,2) on the left and (3+5,5) on the right
# is not symplectic on the full genus-2 surface: the two alpha cycles still
# intersect in the disc-frame intersection form. The basis below is the
# simplest block-supported pair I found that does validate.
rg2_deg = rgs[1]
custom_cycles_deg = [
    {'alpha': [(2, -1)], 'beta': [(0, 1), (2, 1)]},
    {'alpha': [(3, -1), (4, 1)], 'beta': [(4, -1)]},
]

validated_cycles_deg = et.validate_edge_homology_basis(
    et.edge_chord_intersection_matrix(rg2_deg),
    custom_cycles_deg,
    expected_genus=2,
)
print('custom_cycles_deg =', validated_cycles_deg)
print()

x_values = [128,256,512]
omegas = []
for x in x_values:
    ell_deg = [10*3, 7*3, 5*3, 10*3, 7*3, 5*3, x*3, x*3, x*3]
    forms_deg = et.make_cyl_eqn_improved_higher_genus(rg2_deg, ell_deg)
    Omega_deg = et.period_matrix(
        forms=forms_deg,
        ribbon_graph=rg2_deg,
        ell_list=ell_deg,
        custom_cycles=custom_cycles_deg,
    )
    omegas.append(Omega_deg)
    print('x =', x)
    print('ell_deg =', ell_deg)
    print('Omega =')
    print(Omega_deg)
    print('Omega_11 =', Omega_deg[0, 0])
    print('Omega_22 =', Omega_deg[1, 1])
    print('Omega_12 =', Omega_deg[0, 1])
    print('|Omega_12| =', abs(Omega_deg[0, 1]))
    print()

print('Custom-cycle genus-2 degeneration cell ran successfully.')


custom_cycles_deg = [{'alpha': [(2, -1)], 'beta': [(0, 1), (2, 1)]}, {'alpha': [(3, -1), (4, 1)], 'beta': [(4, -1)]}]

x = 128
ell_deg = [30, 21, 15, 30, 21, 15, 384, 384, 384]
Omega =
[[-1.8158376 +0.43213461j  0.22160413+0.13857225j]
 [ 0.22160785+0.13857213j -0.64363771+1.97282657j]]
Omega_11 = (-1.8158376012034925+0.43213460976565005j)
Omega_22 = (-0.6436377064498933+1.9728265681866537j)
Omega_12 = (0.22160412935161555+0.138572251050694j)
|Omega_12| = 0.2613630786988555

x = 256
ell_deg = [30, 21, 15, 30, 21, 15, 768, 768, 768]
Omega =
[[-1.85620522+0.38772446j  0.1938495 +0.13702819j]
 [ 0.19385272+0.13702841j -0.6607425 +2.29067642j]]
Omega_11 = (-1.8562052157207802+0.3877244570744661j)
Omega_22 = (-0.6607424951453479+2.290676424377329j)
Omega_12 = (0.19384949994416703+0.13702819103139324j)
|Omega_12| = 0.23739071962892655



In [36]:
periodsTestAgainst =et.periods_improved((10+7+5)*3*2,10*3,7*3)
periodsTestAgainst[1]/periodsTestAgainst[0]

np.complex128(0.42648579016092514+1.0093782631691806j)

In [10]:
%%timeit
et.periods_improved(16*500,4*500,3*500)

9.84 s ± 157 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
import time
time1 = time.time()
et.periods_improved(16*1000,4*100,3*1000)
time2 = time.time()-time1
print(time2)

85.72816205024719


In [17]:
%%timeit
pf.prime_det_log(1600,400,300)

998 ms ± 18.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
print(pf.prime_det_log_fast(1600,400,300))
print(pf.prime_det_log(1600,400,300))

8.0552446873996188840653404620326298467273024790367
8.0552446873995496061486038522645074121112136118492


In [26]:
%%timeit
pf.prime_det_log_fast(1600,400,300)

The slowest run took 4.77 times longer than the fastest. This could mean that an intermediate result is being cached.
98.7 ms ± 69.8 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [31]:
%%timeit
pf.prime_det_log(8000,2000,1500)

1.37 s ± 566 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
from line_profiler import LineProfiler

L, l1, l2 = 8000,2000,1500

# Profile prime_det_log and its sub-functions
lp = LineProfiler()
lp.add_function(pf.prime_det_log)
lp.add_function(pf.direct_mat_n_fast)
lp.add_function(pf.direct_red_traced_mat)
lp.add_function(pf.logdet_cholesky)

# Wrap and run multiple iterations
wrapped = lp(pf.prime_det_log)
for _ in range(10):
    wrapped(L, l1, l2)

lp.print_stats()

Timer unit: 1e-09 s

Total time: 11.6042 s
File: /Users/sam/Documents/research/string_notes/strebel_differential/python_code/covariant formalism/python/partition_function.py
Function: prime_det_log at line 305

Line #      Hits         Time  Per Hit   % Time  Line Contents
   305                                           def prime_det_log(L: int, l1: int, l2: int) -> mp.mpf:
   306                                               """
   307                                               Returns log(primeDet) where
   308                                                 primeDet = exp(logdet(A')) / (L/2)
   309                                           
   310                                               Fused implementation that exploits circulant structure of Mat to avoid
   311                                               building the full L×L matrix. Only the kernel (length L) and the
   312                                               half×half reduced matrix are allocated.
   313   

In [16]:
%%timeit
et.periods_improved(1600,400,300)

402 ms ± 39 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Genus-1 regression test for the new nu helpers at fairly large edge lengths.
# This should reproduce the verified genus-1 b/nu machinery to much better than 1e-4.
import importlib
import numpy as np
importlib.reload(et)

test_cases_nu = [
    ('generic', 26 * 162, 3 * 162, 1 * 162),
    ('equal',   6 * 162, 1 * 162, 1 * 162),
]

for label, L_nu, l1_nu, l2_nu in test_cases_nu:
    raw_b = np.array(et.calculate_b(L_nu, l1_nu, l2_nu), dtype=np.complex128)
    raw_nu = np.array(et.calculate_nu(L=L_nu, l1=l1_nu, l2=l2_nu), dtype=np.complex128)
    avg_b = np.array(et.average_b(L_nu, l1_nu, l2_nu, list(raw_b)), dtype=np.complex128)
    avg_nu = np.array(et.average_nu(L=L_nu, l1=l1_nu, l2=l2_nu), dtype=np.complex128)

    pm_data = et.period_matrix(L=L_nu, l1=l1_nu, l2=l2_nu, return_data=True)
    alpha_period = pm_data['A_periods'][0, 0]
    raw_nu_norm = raw_nu / alpha_period
    nu_norm = np.array(et.calculate_nu(L=L_nu, l1=l1_nu, l2=l2_nu, normalize_A=True), dtype=np.complex128)

    raw_diff = np.max(np.abs(raw_b - raw_nu))
    avg_diff = np.max(np.abs(avg_b - avg_nu))
    norm_diff = np.max(np.abs(raw_nu_norm - nu_norm))

    print(label, 'L =', L_nu, 'ell =', (l1_nu, l2_nu, L_nu // 2 - l1_nu - l2_nu))
    print('  max |calculate_b - calculate_nu| =', raw_diff)
    print('  max |average_b - average_nu|     =', avg_diff)
    print('  max |raw_nu/P_A - nu_norm|      =', norm_diff)

    assert raw_diff < 1e-10
    assert avg_diff < 1e-10
    assert norm_diff < 1e-10

print('genus-1 nu regression passed')


In [ ]:
# === NU REGRESSION TEST (GENUS 1, LARGE L) ===
# Re-added explicitly at the end of the notebook so it is easy to find.
# This should reproduce the verified genus-1 b/nu machinery to much better than 1e-4.
import importlib
import numpy as np
importlib.reload(et)

test_cases_nu = [
    ('generic', 26 * 162, 3 * 162, 1 * 162),
    ('equal',   6 * 162, 1 * 162, 1 * 162),
]

for label, L_nu, l1_nu, l2_nu in test_cases_nu:
    raw_b = np.array(et.calculate_b(L_nu, l1_nu, l2_nu), dtype=np.complex128)
    raw_nu = np.array(et.calculate_nu(L=L_nu, l1=l1_nu, l2=l2_nu), dtype=np.complex128)
    avg_b = np.array(et.average_b(L_nu, l1_nu, l2_nu, list(raw_b)), dtype=np.complex128)
    avg_nu = np.array(et.average_nu(L=L_nu, l1=l1_nu, l2=l2_nu), dtype=np.complex128)

    pm_data = et.period_matrix(L=L_nu, l1=l1_nu, l2=l2_nu, return_data=True)
    alpha_period = pm_data['A_periods'][0, 0]
    raw_nu_norm = raw_nu / alpha_period
    nu_norm = np.array(et.calculate_nu(L=L_nu, l1=l1_nu, l2=l2_nu, normalize_A=True), dtype=np.complex128)

    raw_diff = np.max(np.abs(raw_b - raw_nu))
    avg_diff = np.max(np.abs(avg_b - avg_nu))
    norm_diff = np.max(np.abs(raw_nu_norm - nu_norm))

    print(label, 'L =', L_nu, 'ell =', (l1_nu, l2_nu, L_nu // 2 - l1_nu - l2_nu))
    print('  max |calculate_b - calculate_nu| =', raw_diff)
    print('  max |average_b - average_nu|     =', avg_diff)
    print('  max |raw_nu/P_A - nu_norm|      =', norm_diff)

    assert raw_diff < 1e-10
    assert avg_diff < 1e-10
    assert norm_diff < 1e-10

print('genus-1 nu regression passed')


In [ ]:
# === GENUS-2 CONSISTENCY TEST (UNIFORM LARGE-ELL SCALING) ===
# Current status: this is not yet an exact analytic benchmark for the full matter+bc formula.
# What it does test is the currently recoverable part: on a good genus-2 graph,
# the period matrix stays in the Siegel upper half plane and the candidate
# genus-2 local-data package stabilizes rapidly as all edge lengths are scaled up.
import importlib
import numpy as np
importlib.reload(et)

rg2_nu = rgs[1]
scales_g2 = [11, 22, 33]
results_g2 = []

for scale in scales_g2:
    ell_g2 = [scale] * 9
    forms_g2 = et.make_cyl_eqn_improved_higher_genus(rg2_nu, ell_g2)
    data_g2 = et.genus2_matter_bc_candidate(forms_g2, ribbon_graph=rg2_nu, ell_list=ell_g2)

    Omega_g2 = np.asarray(data_g2['Omega'], dtype=np.complex128)
    imag_evals = np.linalg.eigvalsh(np.imag(Omega_g2))
    sym_err = np.max(np.abs(Omega_g2 - Omega_g2.T))
    max_group_err = max(data_g2['group_errors'].values())

    results_g2.append({
        'scale': scale,
        'Omega': Omega_g2,
        'imag_evals': imag_evals,
        'sym_err': sym_err,
        'nu_factor': data_g2['nu_factor'],
        'candidate_factor': data_g2['candidate_factor'],
        'max_group_err': max_group_err,
    })

    print(f'scale = {scale}')
    print('  Omega =')
    print(Omega_g2)
    print('  eig(Im Omega) =', imag_evals)
    print('  symmetry error =', sym_err)
    print('  nu_factor =', data_g2['nu_factor'])
    print('  candidate_factor =', data_g2['candidate_factor'])
    print('  max group proportionality error =', max_group_err)

for i in range(len(results_g2) - 1):
    r_lo = results_g2[i]
    r_hi = results_g2[i + 1]
    dOmega = np.max(np.abs(r_hi['Omega'] - r_lo['Omega']))
    dCand = abs(r_hi['candidate_factor'] - r_lo['candidate_factor'])
    print(f'delta Omega ({r_lo["scale"]} -> {r_hi["scale"]}) =', dOmega)
    print(f'delta candidate_factor ({r_lo["scale"]} -> {r_hi["scale"]}) =', dCand)

# The strict benchmark we currently trust is internal convergence, not an exact closed form.
assert all(np.min(r['imag_evals']) > 0.0 for r in results_g2)
assert all(r['sym_err'] < 5e-4 for r in results_g2)
assert results_g2[0]['max_group_err'] > results_g2[1]['max_group_err'] > results_g2[2]['max_group_err']
assert np.max(np.abs(results_g2[2]['Omega'] - results_g2[1]['Omega'])) < 1e-4
assert abs(results_g2[2]['candidate_factor'] - results_g2[1]['candidate_factor']) < 1e-6

print('genus-2 consistency test passed')


In [6]:
# === BAD GENUS-2 OMEGA EXAMPLE ===
# This is the explicit built-in graph / length choice currently known to produce
# a non-symmetric Omega with Im(Omega) not positive definite.
import importlib
import numpy as np
import ell_to_tau as et
importlib.reload(et)
from ribbon_graph_generator import generate_ribbon_graphs, get_disc_boundary
#rgs = generate_ribbon_graphs(n_faces=1, n_edges=9)

rg2_bad = rgs[0]
ell_bad = [110] * 9
forms_bad = et.make_cyl_eqn_improved_higher_genus(rg2_bad, ell_bad)
pm_bad = et.period_matrix(forms=forms_bad, ribbon_graph=rg2_bad, ell_list=ell_bad, return_data=True)
Omega_bad = np.asarray(pm_bad['Omega'], dtype=np.complex128)

print('Omega_bad =')
print(Omega_bad)
print('max |Omega - Omega^T| =', np.max(np.abs(Omega_bad - Omega_bad.T)))
print('eig(Im Omega) =', np.linalg.eigvalsh(np.imag(Omega_bad)))


V=6, E=9, F=1, g=2
Using 2 base cubic graph(s)
Total valid rotation systems: 34992
  Base graph (6V, 9E): 72 automorphisms, 17496 rotation systems
  Base graph (6V, 9E): 12 automorphisms, 17496 rotation systems
Non-isomorphic ribbon graphs: 4
Omega_bad =
[[ 0.51859353+1.03169476j -0.25929705-0.51584795j]
 [-0.25929648-0.51584682j  0.51859353+1.03169476j]]
max |Omega - Omega^T| = 1.2631839723559354e-06
eig(Im Omega) = [0.51584795 1.54754158]


In [ ]:
# === EQUAL-LENGTH GENUS-2 SCAN ACROSS BUILT-IN GRAPHS ===
# This helps separate three issues:
# 1. whether Omega is even admissible (symmetric with Im(Omega)>0),
# 2. whether equal lengths on this graph look symmetry-enhanced, and
# 3. whether the mismatch is graph-dependent rather than a universal period_matrix failure.
import importlib
import numpy as np
importlib.reload(et)

ell_equal_g2 = [11] * 9
for idx, rg_equal in enumerate(rgs):
    print(f'graph {idx}')
    try:
        forms_equal = et.make_cyl_eqn_improved_higher_genus(rg_equal, ell_equal_g2)
        pm_equal = et.period_matrix(forms=forms_equal, ribbon_graph=rg_equal, ell_list=ell_equal_g2, return_data=True)
        Omega_equal = np.asarray(pm_equal['Omega'], dtype=np.complex128)
        sym_err = np.max(np.abs(Omega_equal - Omega_equal.T))
        imag_evals = np.linalg.eigvalsh(np.imag(Omega_equal))
        diag_diff = abs(Omega_equal[0, 0] - Omega_equal[1, 1])
        print('  basis =', pm_equal['basis_pairs'])
        print('  Omega =')
        print(Omega_equal)
        print('  max |Omega - Omega^T| =', sym_err)
        print('  eig(Im Omega) =', imag_evals)
        print('  |Omega_11 - Omega_22| =', diag_diff)
    except Exception as exc:
        print('  ERROR:', type(exc).__name__, exc)
    print()


In [4]:
# === GENUS-2 NUMERICAL DETERMINANT VS ANALYTIC CANDIDATE (EVEN PARITY SECTOR) ===
# Current status: the graph-based determinant side is numerically reliable in the even-scaling sector
# on rgs[1]. Odd overall scalings show a parity pathology in the bc determinant and are not tested here.
import importlib
import partition_function as pf
import ell_to_tau as et
from Fast_Ribbon_Generator import generate_ribbon_graphs
import mpmath as mp
import numpy as np

importlib.reload(pf)
importlib.reload(et)

rgs = generate_ribbon_graphs(1, 9)
rg2_det = rgs[1]
ref_base = [12, 10, 11, 11, 12, 10, 11, 11, 11]
num_base = [13, 9, 11, 11, 13, 9, 11, 11, 11]
assert sum(ref_base) == sum(num_base), (
    'This determinant-ratio benchmark only cancels the unknown lattice normalization '
    'at fixed total reduced boundary length: require sum(num_base) == sum(ref_base).'
)
scales_det = [2, 4, 6,8,10,20]

det_results = []
for s in scales_det:
    ell_ref = [s * x for x in ref_base]
    ell_num = [s * x for x in num_base]

    log_ref = pf.traced_numeric_amplitude_log_psi1_f1(rg2_det, ell_ref)
    log_num = pf.traced_numeric_amplitude_log_psi1_f1(rg2_det, ell_num)
    numeric_ratio = complex(mp.e ** (log_num - log_ref))

    forms_ref = et.make_cyl_eqn_improved_higher_genus(rg2_det, ell_ref)
    data_ref = et.genus2_matter_bc_candidate(forms_ref, ribbon_graph=rg2_det, ell_list=ell_ref)
    forms_num = et.make_cyl_eqn_improved_higher_genus(rg2_det, ell_num)
    data_num = et.genus2_matter_bc_candidate(forms_num, ribbon_graph=rg2_det, ell_list=ell_num)
    analytic_ratio = data_num['candidate_factor'] / data_ref['candidate_factor']

    rel_diff = abs(numeric_ratio - analytic_ratio) / abs(analytic_ratio)
    det_results.append((s, sum(ell_ref), numeric_ratio, analytic_ratio, rel_diff))

    print(f'scale = {s}, total reduced boundary length = {sum(ell_ref)}')
    print('  numeric ratio  =', numeric_ratio)
    print('  analytic ratio =', analytic_ratio)
    print('  rel diff       =', rel_diff)

for s, total_len, numeric_ratio, analytic_ratio, rel_diff in det_results:
    assert rel_diff < 1e-3, (s, total_len, rel_diff)


scale = 2, total reduced boundary length = 198
  numeric ratio  = (1.1971891637612737+0j)
  analytic ratio = 1.1060446663267214
  rel diff       = 0.08240580169085919
scale = 4, total reduced boundary length = 396
  numeric ratio  = (1.1992476285615823+0j)
  analytic ratio = 1.106024767003615
  rel diff       = 0.08428641413747169
scale = 6, total reduced boundary length = 594
  numeric ratio  = (1.200020410025319+0j)
  analytic ratio = 1.1060217091158748
  rel diff       = 0.08498811563525656


KeyboardInterrupt: 

In [5]:
# === GENUS-2 NUMERICAL DETERMINANT VS ANALYTIC CANDIDATE (EVEN PARITY SECTOR) ===
# Current status: the graph-based determinant side is numerically reliable in the even-scaling sector
# on rgs[1]. Odd overall scalings show a parity pathology in the bc determinant and are not tested here.
import importlib
import partition_function as pf
import ell_to_tau as et
from Fast_Ribbon_Generator import generate_ribbon_graphs
import mpmath as mp
import numpy as np

importlib.reload(pf)
importlib.reload(et)

rgs = generate_ribbon_graphs(1, 9)
rg2_det = rgs[1]
ref_base = [12, 10, 11, 11, 12, 10, 11, 11, 11]
num_base = [19, 5, 11, 11, 17, 3, 11, 11, 11]
assert sum(ref_base) == sum(num_base), (
    'This determinant-ratio benchmark only cancels the unknown lattice normalization '
    'at fixed total reduced boundary length: require sum(num_base) == sum(ref_base).'
)
scales_det = [2, 4, 6,8,10,20]

det_results = []
for s in scales_det:
    ell_ref = [s * x for x in ref_base]
    ell_num = [s * x for x in num_base]

    log_ref = pf.traced_numeric_amplitude_log_psi1_f1(rg2_det, ell_ref)
    log_num = pf.traced_numeric_amplitude_log_psi1_f1(rg2_det, ell_num)
    numeric_ratio = complex(mp.e ** (log_num - log_ref))

    forms_ref = et.make_cyl_eqn_improved_higher_genus(rg2_det, ell_ref)
    data_ref = et.genus2_matter_bc_candidate(forms_ref, ribbon_graph=rg2_det, ell_list=ell_ref)
    forms_num = et.make_cyl_eqn_improved_higher_genus(rg2_det, ell_num)
    data_num = et.genus2_matter_bc_candidate(forms_num, ribbon_graph=rg2_det, ell_list=ell_num)
    analytic_ratio = data_num['candidate_factor'] / data_ref['candidate_factor']

    rel_diff = abs(numeric_ratio - analytic_ratio) / abs(analytic_ratio)
    det_results.append((s, sum(ell_ref), numeric_ratio, analytic_ratio, rel_diff))

    print(f'scale = {s}, total reduced boundary length = {sum(ell_ref)}')
    print('  numeric ratio  =', numeric_ratio)
    print('  analytic ratio =', analytic_ratio)
    print('  rel diff       =', rel_diff)

for s, total_len, numeric_ratio, analytic_ratio, rel_diff in det_results:
    assert rel_diff < 1e-3, (s, total_len, rel_diff)


scale = 2, total reduced boundary length = 198
  numeric ratio  = (2.413495167794677+0j)
  analytic ratio = 3.8257473448391917
  rel diff       = 0.36914413047939426
scale = 4, total reduced boundary length = 396
  numeric ratio  = (2.4407918353800473+0j)
  analytic ratio = 3.824680504674511
  rel diff       = 0.3618311823963021


KeyboardInterrupt: 